# Spectrogram Generation

In this notebook, the previously labeled vibration time-series segments are transformed into time–frequency representations using short-time Fourier transform (STFT). The resulting spectrograms are normalized and converted into RGB images by stacking the X, Y, and Z acceleration axes. These images serve as input for the subsequent convolutional neural network (CNN) used for chatter detection.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.signal import spectrogram
from PIL import Image
import os
from tqdm import tqdm
import random
import h5py

# In-/ Output Structure & Class-Mapping

In [2]:
INPUT_ROOT = Path(r"C:\Users\Zeleny\Documents\01_VSCode\cnn-anomaly-net\cnn-anomaly-net\data\raw")
OUTPUT_ROOT = Path("../../data/04_bosch_dataset")

# Spectrogram configuration
- NPERSEG: controls frequency resolution of spectrogram
larger values → better frequency resolution, lower time resolution
- MAX_FREQ_CUT: limits analysis to relevant chatter frequency range
- DB_MIN / DB_MAX: spectrogram is converted to decibel scale --> fixed scaling for energy
- TARGET_IMG_SIZE: image size of three color spectrogram

In [3]:
# segmentation of timeseries  
WINDOW_SEC = 5    

# config of spectrogram
NPERSEG = 512
FS = 2000                 # sampling
MAX_FREQ_CUT = 2000      # max. freq.
DB_MIN = -55             
DB_MAX = -6              
EPS = 1e-12

# output image size
TARGET_IMG_SIZE = (150, 100) 

WINDOW_SIZE = int(WINDOW_SEC * FS)

## Spectrogram generator function

In [4]:
# =========================================================
# LOAD DATA
# =========================================================
def load_data(file_path):
    with h5py.File(file_path, 'r') as f:
        return f['vibration_data'][:]

# =========================================================
# SPECTROGRAM
# =========================================================
def create_spectrogram(sig, fs):
    noverlap = int(0.75 * NPERSEG)

    f, t, Sxx = spectrogram(
        sig,
        fs=fs,
        nperseg=NPERSEG,
        noverlap=noverlap,
        mode="psd"
    )

    # Frequency limitation
    mask = f <= MAX_FREQ_CUT
    Sxx = Sxx[mask, :]

    # dB-Scaling
    Sxx_db = 10 * np.log10(Sxx + EPS)
    Sxx_db = np.clip(Sxx_db, DB_MIN, DB_MAX)

    # Normalization
    Sxx_norm = 1.0 - (Sxx_db - DB_MIN) / (DB_MAX - DB_MIN)

    return Sxx_norm

def process_file(file_path):
    data = load_data(file_path)

    x = data[:, 0] / 1000
    y = data[:, 1] / 1000
    z = data[:, 2] / 1000

    x_segments = segment_signal(x)
    y_segments = segment_signal(y)
    z_segments = segment_signal(z)

    relative_path = file_path.relative_to(INPUT_ROOT)

    for i, (xs, ys, zs) in enumerate(zip(x_segments, y_segments, z_segments)):

        specs = [
            create_spectrogram(xs, FS),
            create_spectrogram(ys, FS),
            create_spectrogram(zs, FS)
        ]

        rgb = np.stack(specs, axis=-1)
        rgb = np.clip(rgb, 0, 1)
        rgb = np.flipud(rgb)

        img = Image.fromarray((rgb * 255).astype(np.uint8))
        img = img.resize(TARGET_IMG_SIZE)

        # Ordner basierend auf Dateinamen erstellen
        file_output_dir = OUTPUT_ROOT / relative_path.parent / file_path.stem

        # kompletter Pfad inkl. Dateiname
        out_path = file_output_dir / f"{file_path.stem}_seg{i}.png"

        # Ordner sicher erstellen
        out_path.parent.mkdir(parents=True, exist_ok=True)
        img.save(out_path)


def segment_signal(sig):
    segments = []
    
    for start in range(0, len(sig) - WINDOW_SIZE, WINDOW_SIZE):
        segment = sig[start:start + WINDOW_SIZE]
        segments.append(segment)
    
    return segments


In [5]:
def create_spectrogram(sig, fs):
    noverlap = int(0.75 * NPERSEG)

    f, t, Sxx = spectrogram(
        sig,
        fs=fs,
        nperseg=NPERSEG,
        noverlap=noverlap,
        mode="psd"
    )

    mask = f <= MAX_FREQ_CUT
    Sxx = Sxx[mask, :]

    Sxx_db = 10 * np.log10(Sxx + EPS)
    Sxx_db = np.clip(Sxx_db, DB_MIN, DB_MAX)

    return 1.0 - (Sxx_db - DB_MIN) / (DB_MAX - DB_MIN)

# RGB Image Conversion
This function converts the labeled vibration time-series segment into a 3-channel spectrogram image.
Combines three spectrograms into a single 3-channel image:

R = X-axis
G = Y-axis
B = Z-axis

Each vibration segment is transformed into a three-channel spectrogram image, where each channel corresponds to a different sensor axis. This encoding preserves spatial vibration directionality while enabling convolutional neural networks to learn cross-axis correlations relevant for chatter detection.

**Advantages of these representation:**
- harmonic stripes (chatter)
- broadband noise (instability)
- axis coupling effects

In [7]:
all_files = list(INPUT_ROOT.rglob("*.h5"))

print(f"Found {len(all_files)} files")

for f in all_files:
    try:
        process_file(f)
    except Exception as e:
        print(f"Error with {f}: {e}")


Found 1702 files
